# Financial News Sentiment Analysis Using FinBERT — actual fine-tuning

Akshat Agarwal · 202401100100028 · CS-5A · KIET Group of Institutions, Ghaziabad

**Before running:** Runtime → Change runtime type → GPU (T4 if available).
Then choose Runtime → Run all and upload `finbert_colab_training.zip` when asked.
No API key, GitHub token, or training code changes are required.

This trains a new sentiment classifier and the encoder of **HKUST
`yiyanghkust/finbert-pretrain`**, a different FinBERT release from the app's
existing Prosus reference. Only train/validation data is used during training.
The held-out test is evaluated after the best checkpoint is selected.
The short smoke test is a technical check and has no final accuracy claim.

Save the downloaded **finbert_finetuned_results.zip** in `D:\Desktop\CA2_DL`.
It contains the trained model, tokenizer, settings, logs, metrics, and figures.
GPU availability and runtime lifetime are controlled by Colab; download results
before disconnecting. Optional Drive backup is explained in the final notes.


## 1. Upload the private input bundle


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, subprocess, sys, zipfile
from google.colab import files

EXPECTED_SHA256 = "0b601b444afd6c94457e7c991948e077b82135192949144b4a2fd39cc4dff264"
uploaded = files.upload()
if set(uploaded) != {"finbert_colab_training.zip"}:
    raise ValueError("Upload only finbert_colab_training.zip from the project .cache/colab folder.")
payload = uploaded.pop("finbert_colab_training.zip")
if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Bundle mismatch. Use the ZIP that accompanies this notebook.")
ROOT = Path("/content/financial_sentiment_training")
if ROOT.exists() and any(ROOT.iterdir()):
    raise FileExistsError("This runtime already has an extracted project. Continue with the next cell; do not replace it.")
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json"))
    expected = set(manifest["files"]) | {"bundle_manifest.json"}
    if set(archive.namelist()) != expected or len(archive.namelist()) != len(expected):
        raise ValueError("Unexpected ZIP members.")
    ROOT.mkdir(parents=True, exist_ok=True)
    for name, expected_hash in manifest["files"].items():
        member = PurePosixPath(name)
        if member.is_absolute() or ".." in member.parts or "\\" in name or ":" in name:
            raise ValueError("Unsafe ZIP member.")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected_hash:
            raise ValueError(f"File checksum mismatch: {name}")
        destination = ROOT.joinpath(*member.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)
    (ROOT / "bundle_manifest.json").write_text(json.dumps(manifest, indent=2))
del payload, uploaded
os.chdir(ROOT)
print("Verified input bundle. Exact saved splits: train 3385 / validation 725 / test 726.")


## 2. Install training libraries and verify GPU


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-training.txt"], check=True)
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'No CUDA GPU: change runtime type to GPU, then rerun'; print('GPU:', torch.cuda.get_device_name(0)); print('PyTorch:', torch.__version__); print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))"], check=True)


## 3. Check real training, validation, and checkpoint saving

This short run uses only training/validation subsets. It must finish successfully
before the main run. It does not produce a model for the submitted app.
Warnings about newly initialized classifier/pooler weights and unused pretraining
head weights are expected. A traceback or failed cell needs attention.


In [ ]:
subprocess.run([sys.executable, "scripts/train_finbert.py", "--smoke-test"], check=True)


## 4. Fine-tune on all training rows

Full encoder + classifier fine-tuning, seed 42, 3 epochs, maximum 128 tokens,
learning rate 2e-5, microbatch 2, gradient accumulation 8 (effective batch 16 on
one GPU), mixed precision, dynamic padding, and gradient checkpointing.
Select the saved checkpoint with the best **validation macro-F1**.

Do not change settings based on test results. A new run uses a new output folder.
If this cell is interrupted, preserve any saved checkpoints before disconnecting
and ask for help resuming rather than repeating the test.


In [ ]:
MODEL_DIR = "models/finbert_finetuned"
subprocess.run([sys.executable, "scripts/train_finbert.py", "--output-dir", MODEL_DIR], check=True)


## 5. Evaluate the selected frozen checkpoint

The 726 test sentences are now used for evaluation, after training/selection.
Save accuracy, macro-F1, per-class results, and the confusion matrix. Compare
with our TF-IDF baseline on the exact same test membership. The older Prosus
score remains an overlapping-data diagnostic because Prosus was already
fine-tuned on Financial PhraseBank. This experiment does not prove that every
test sentence was absent from HKUST's large pretraining corpus.


In [ ]:
subprocess.run([sys.executable, "scripts/evaluate_finetuned.py", "--model-dir", MODEL_DIR, "--output-dir", "outputs/finetuning"], check=True)


## 6. Export and download the result bundle


In [ ]:
RESULT_ZIP = ROOT / "finbert_finetuned_results.zip"
if not RESULT_ZIP.exists():
    subprocess.run([sys.executable, "scripts/export_finetuned.py", "--model-dir", MODEL_DIR, "--evaluation-dir", "outputs/finetuning", "--destination", str(RESULT_ZIP)], check=True)
with zipfile.ZipFile(RESULT_ZIP) as archive:
    if archive.testzip() is not None:
        raise ValueError("The export ZIP is damaged; preserve the model and ask for help exporting again.")
print("Save this ZIP in D:\\Desktop\\CA2_DL. Keep its model and manifest together.")
files.download(str(RESULT_ZIP))


## After downloading

Tell Codex the local ZIP path so it can verify hashes, inspect the training
evidence, load the saved classifier, and update the application, README and PPT
with measured results. Preparation or smoke-test success is not completed
FinBERT fine-tuning. Do not claim improvement until the final metrics exist.

If browser downloads are blocked, allow the download and rerun only the last
cell. Colab VM files can disappear when the runtime is deleted. For a longer run,
you can optionally mount Drive and use `--checkpoint-dir` pointing to a new
Drive folder before training; Drive authorization is your choice. Epoch
checkpoints are excluded from the final download to keep it smaller, so preserve
them separately if you need resumable optimizer state.

Sources: [HKUST checkpoint](https://huggingface.co/yiyanghkust/finbert-pretrain),
[authors' fine-tuning example](https://github.com/yya518/FinBERT/blob/master/finetune.ipynb),
[Colab resource/lifetime information](https://research.google.com/colaboratory/faq.html).
